In [99]:
import os
import io
import tempfile
import warnings
 
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import stackstac
import pystac_client
import planetary_computer
import torch
import torch.nn as nn
from rasterio.features import rasterize
from scipy.stats import rankdata, spearmanr, pearsonr
from sklearn.metrics import f1_score, jaccard_score, confusion_matrix
from sklearn.metrics import average_precision_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle
from matplotlib import font_manager
from sklearn.metrics import precision_recall_curve, average_precision_score
from scipy.stats import spearmanr

In [95]:
warnings.filterwarnings("ignore")
 
TIGER = "https://www2.census.gov/geo/tiger/TIGER2024/PLACE/tl_2024_%s_place.zip"
TIGER_TRACT = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_%s_tract.zip"
STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
LNDCOV_WMS = "https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Native_conus_year_data/wms"
LNDCOV_LAYER = "Land-Cover-Native_conus_year_data"
CARTO = "https://phl.carto.com/api/v2/sql"
VPI_HUB = ("https://hub.arcgis.com/api/v3/datasets/"
           "f7ed68293c5e40d58f1de9c8435c3e84_0/downloads/data"
           "?format=geojson&spatialRefId=4326&where=1%3D1")
 
CITIES = {"Philadelphia": ("42", "4260000"),
          "Detroit": ("26", "2622000"),
          "Atlanta": ("13", "1304000")}
STATE_OF = {"Philadelphia": "42", "Detroit": "26", "Atlanta": "13"}
HOME = "Philadelphia"
T0, T1 = 2015, 2025
EARLY = [2015, 2016, 2017]
LATE = [2023, 2024, 2025]
ALL_YEARS = sorted(set(EARLY + LATE))
 
BANDS = ["blue", "green", "red", "nir08", "swir16", "swir22", "lwir11"]
NDVI_T0, NDBI_T0, NDVI_T1, NDBI_T1 = 7, 8, 16, 17
DEVELOPED = (21, 22, 23, 24)
INTENSITY = {21: 1, 22: 2, 23: 3, 24: 4}
WATER = (11, 12)
WETLAND = (90, 95)
NLCD_RGB = {(70, 107, 159): 11, (209, 222, 248): 12, (222, 197, 197): 21,
            (217, 146, 130): 22, (235, 0, 0): 23, (171, 0, 0): 24,
            (179, 172, 159): 31, (104, 171, 95): 41, (28, 95, 44): 42,
            (181, 197, 143): 43, (204, 184, 121): 52, (223, 223, 194): 71,
            (220, 217, 57): 81, (171, 108, 40): 82, (184, 217, 235): 90,
            (108, 159, 184): 95}

STALE = ["fig3_model_vs_null.png", "fig6_development_regime.png",
         "fig5_validation_panels.png", "scale_sensitivity.png",
         "model_vs_null.png", "detected_change.png",
         "philadelphia_observed_vs_detected.png", "validation_vacancy.png"]

PIXEL_M = 30
TILE = 1800
PATCH = 128
STRIDE = 64
BLOCK_PX = 256
EPOCHS = 40
BATCH = 16
SEED = 650
CELL_SIZES = [90, 150, 300, 600, 1200, 2400]
PRIMARY_CELL = 300

In [96]:
INK = "#381f04"
PAPER = "#f1f2e0"
MUTE = "#5a544c"
GOLD = "#dfad10"
LIME = "#bad012"
SAGE = "#9ab0a6"
UMBER = "#68542d"
SAND = "#dfd78b"
STONE = "#e9e7e0"
WHITE = "#ffffff"
DOT = "#c9c4ab"
TRACT_LINE = "#9ab0a6"
PANEL_TINTS = ["#ffffff", "#ffffff", "#ffffff"]
SEQUENTIAL = ["#ffffff", "#dfd78b", "#dfad10", "#68542d", "#381f04"]
SERIES = [UMBER, GOLD, SAGE, MUTE, LIME]
 
DECAY_C = GOLD
NEWDEV_C = LIME
STROKE = 2.6
SLIVER_M2 = 10000
DOT_SPACING_M = 420
DOT_SIZE = 1.5
 
available = {f.name for f in font_manager.fontManager.ttflist}
for wanted in ["Archivo Black", "Anton", "Helvetica Neue", "Helvetica",
               "Arial Black", "Arial", "DejaVu Sans"]:
    if wanted in available:
        FONT = wanted
        break
print("title font: %s" % FONT)

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "font.family": FONT, "text.color": INK,
    "axes.edgecolor": INK, "axes.linewidth": STROKE,
    "axes.labelcolor": INK, "axes.labelsize": 12, "axes.labelweight": "bold",
    "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "xtick.major.width": STROKE, "ytick.major.width": STROKE,
    "legend.frameon": False,
    "figure.dpi": 400, "savefig.dpi": 400,
    "savefig.transparent": False, "figure.edgecolor": PAPER,
})

title font: Helvetica Neue


In [70]:
import tempfile
os.chdir("/Users/cyberhbliu/Desktop/PERSONAL/2026portfolio/urban_decay_and_sprawl")
os.makedirs("outputs", exist_ok=True)
OUT_ROOT = os.path.join("outputs", "urban_decay_and_sprawl")
print("writing to %s" % OUT_ROOT)

writing to outputs/urban_decay_and_sprawl


# 1. fetching NLCD labels over WMS

In [71]:
GRID, COVER = {}, {}
for city, (state, place) in CITIES.items():
    shape = gpd.read_file(TIGER % state)
    shape = shape[shape["GEOID"] == place].to_crs("EPSG:5070")
    minx, miny, maxx, maxy = shape.total_bounds
    minx = np.floor(minx / PIXEL_M) * PIXEL_M
    maxy = np.ceil(maxy / PIXEL_M) * PIXEL_M
    width = int(np.ceil((maxx - minx) / PIXEL_M))
    height = int(np.ceil((maxy - miny) / PIXEL_M))
    maxx, miny = minx + width * PIXEL_M, maxy - height * PIXEL_M
    transform = rasterio.transform.from_origin(minx, maxy, PIXEL_M, PIXEL_M)
    inside = rasterize([(x, 1) for x in shape.geometry], out_shape=(height, width),
                       transform=transform, fill=0, dtype="uint8").astype(bool)
    GRID[city] = {"shape": shape, "width": width, "height": height,
                  "bounds": (minx, miny, maxx, maxy), "transform": transform,
                  "inside": inside}
 
    for year in ALL_YEARS:
        canvas = None
        for r0 in range(0, height, TILE):
            for c0 in range(0, width, TILE):
                h, w = min(TILE, height - r0), min(TILE, width - c0)
                bx0, by1 = minx + c0 * PIXEL_M, maxy - r0 * PIXEL_M
                resp = requests.get(LNDCOV_WMS, params={
                    "service": "WMS", "version": "1.1.1", "request": "GetMap",
                    "layers": LNDCOV_LAYER, "styles": "", "srs": "EPSG:5070",
                    "bbox": "%f,%f,%f,%f" % (bx0, by1 - h * PIXEL_M,
                                             bx0 + w * PIXEL_M, by1),
                    "width": w, "height": h, "format": "image/geotiff",
                    "transparent": "false",
                    "time": "%d-01-01T00:00:00.000Z" % year}, timeout=300)
                resp.raise_for_status()
                if b"ServiceException" in resp.content[:2000]:
                    raise SystemExit(resp.content[:600].decode("utf-8", "ignore"))
                with rasterio.MemoryFile(resp.content) as mem:
                    with mem.open() as src:
                        block = src.read()
                if canvas is None:
                    canvas = np.zeros((block.shape[0], height, width), block.dtype)
                canvas[:, r0:r0 + h, c0:c0 + w] = block[:, :h, :w]
 
        if canvas.shape[0] == 1:
            arr = canvas[0].astype(np.uint8)
        else:
            arr = np.zeros((height, width), np.uint8)
            rgb = canvas[:3].transpose(1, 2, 0)
            for colour, code in NLCD_RGB.items():
                arr[np.all(rgb == np.array(colour, canvas.dtype), axis=-1)] = code
            if (arr[inside] > 0).mean() < 0.98:
                raise SystemExit("legend decode failed for %s %d" % (city, year))
        COVER[(city, year)] = arr
    print("      %-13s %d x %d, %d years" % (city, height, width, len(ALL_YEARS)))

      Philadelphia  1109 x 818, 6 years
      Detroit       803 x 1032, 6 years
      Atlanta       903 x 791, 6 years


# 2. building imagery and labels

In [72]:
catalog = pystac_client.Client.open(STAC, modifier=planetary_computer.sign_inplace)
DATA = {}
 
for city in CITIES:
    g = GRID[city]
    height, width = g["height"], g["width"]
    inside, transform = g["inside"], g["transform"]
    minx, miny, maxx, maxy = g["bounds"]
 
    ranks = {}
    for year in ALL_YEARS:
        r = np.zeros((height, width), np.int8)
        for code, level in INTENSITY.items():
            r[COVER[(city, year)] == code] = level
        ranks[year] = r
    early = np.median(np.stack([ranks[y] for y in EARLY]), axis=0)
    late = np.median(np.stack([ranks[y] for y in LATE]), axis=0)
 
    cov0 = COVER[(city, T0)]
    blocked = np.isin(cov0, WATER) | np.isin(cov0, WETLAND)
    developed_early = early >= 1
    convertible = (early == 0) & (~blocked)
 
    label = np.zeros((height, width), np.int64)
    label[convertible & (late >= 1)] = 1
    label[developed_early & (late < early)] = 2
    label[~inside] = 0
 
    wgs = g["shape"].to_crs("EPSG:4326").total_bounds
    scenes = []
    for year in (T0, T1):
        items = catalog.search(collections=["landsat-c2-l2"], bbox=list(wgs),
                               datetime="%d-05-01/%d-09-30" % (year, year),
                               query={"eo:cloud_cover": {"lt": 30},
                                      "platform": {"in": ["landsat-8", "landsat-9"]}}
                               ).item_collection()
        if len(items) == 0:
            raise SystemExit("no Landsat scenes for %s %d" % (city, year))
        cube = stackstac.stack(items, assets=BANDS + ["qa_pixel"], epsg=5070,
                               resolution=PIXEL_M, bounds=(minx, miny, maxx, maxy),
                               chunksize=1024, dtype="float32",
                               fill_value=np.float32("nan"), rescale=False)
        qa = cube.sel(band="qa_pixel").fillna(0).astype("uint16")
        clear = ((qa & (1 << 1)) == 0) & ((qa & (1 << 3)) == 0) & ((qa & (1 << 4)) == 0)
        composite = cube.sel(band=BANDS).where(clear).median("time").compute().values
        optical = composite[:6] * 0.0000275 - 0.2
        thermal = (composite[6] * 0.00341802 + 149.0 - 273.15) / 50.0
        nd = lambda a, b: (a - b) / (a + b + 1e-6)
        scenes.append(np.concatenate([optical, thermal[None],
                                      nd(optical[3], optical[2])[None],
                                      nd(optical[4], optical[3])[None]]).astype(np.float32))
        print("      %-13s %d  %d scenes" % (city, year, len(items)))
 
    image = np.nan_to_num(np.concatenate(scenes), nan=0.0)
    if image.shape[1:] != (height, width):
        pad = np.zeros((image.shape[0], height, width), np.float32)
        h, w = min(height, image.shape[1]), min(width, image.shape[2])
        pad[:, :h, :w] = image[:, :h, :w]
        image = pad
 
    rows, cols = np.mgrid[0:height, 0:width]
    DATA[city] = {"image": image, "label": label, "inside": inside,
                  "developed_early": developed_early & inside,
                  "convertible": convertible & inside,
                  "checker": ((rows // BLOCK_PX) + (cols // BLOCK_PX)) % 2,
                  "shape": g["shape"], "transform": transform,
                  "extent": (minx, maxx, miny, maxy)}
    print("      %-13s new development %d  decay %d"
          % (city, (label == 1).sum(), (label == 2).sum()))

      Philadelphia  2015  4 scenes
      Philadelphia  2025  6 scenes
      Philadelphia  new development 1426  decay 4424
      Detroit       2015  5 scenes
      Detroit       2025  27 scenes
      Detroit       new development 103  decay 11211
      Atlanta       2015  11 scenes
      Atlanta       2025  13 scenes
      Atlanta       new development 3010  decay 6124


# 3. cutting patches

In [73]:
train = []
d = DATA[HOME]
for r in range(0, d["label"].shape[0] - PATCH + 1, STRIDE):
    for c in range(0, d["label"].shape[1] - PATCH + 1, STRIDE):
        m = d["inside"][r:r + PATCH, c:c + PATCH]
        if m.mean() < 0.5 or d["checker"][r:r + PATCH, c:c + PATCH].mean() != 0:
            continue
        train.append((d["image"][:, r:r + PATCH, c:c + PATCH],
                      d["label"][r:r + PATCH, c:c + PATCH], m))
 
MU = np.stack([p[0] for p in train]).mean(axis=(0, 2, 3))
SD = np.stack([p[0] for p in train]).std(axis=(0, 2, 3)) + 1e-6
Xt = torch.from_numpy((np.stack([p[0] for p in train]) - MU[None, :, None, None])
                      / SD[None, :, None, None]).float()
yt = torch.from_numpy(np.stack([p[1] for p in train]))
mt = torch.from_numpy(np.stack([p[2] for p in train]).astype(np.float32))
print("      %d patches, %d channels" % (Xt.shape[0], Xt.shape[1]))

      28 patches, 18 channels


# 4. training the U-Net

In [74]:
class UNet(nn.Module):
    def __init__(self, ch, out):
        super().__init__()
        block = lambda i, o: nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.e1, self.e2, self.e3 = block(ch, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.d2 = nn.ConvTranspose2d(128, 64, 2, 2), block(128, 64)
        self.u1, self.d1 = nn.ConvTranspose2d(64, 32, 2, 2), block(64, 32)
        self.head = nn.Conv2d(32, out, 1)
 
    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        e = self.d1(torch.cat([self.u1(self.d2(torch.cat([self.u2(c), b], 1))), a], 1))
        return self.head(e)
 
 
counts = np.array([(yt.numpy() == k).sum() for k in range(3)], np.float64) + 1
raw = (counts.sum() / counts) ** 0.5
weights = torch.tensor(raw / raw[0], dtype=torch.float32, device=device)
print("      class weights %s" % np.round(weights.cpu().numpy(), 2))
 
net = UNet(Xt.shape[1], 3).to(device)
opt = torch.optim.Adam(net.parameters(), 1e-3)
ce = nn.CrossEntropyLoss(weight=weights, reduction="none")
ALPHA, BETA = 0.3, 0.7
 
n = Xt.shape[0]
for epoch in range(EPOCHS):
    net.train()
    order = torch.randperm(n)
    total = 0.0
    for s in range(0, n, BATCH):
        i = order[s:s + BATCH]
        xb, yb, mb = Xt[i].to(device), yt[i].to(device), mt[i].to(device)
        opt.zero_grad()
        logits = net(xb)
        loss = (ce(logits, yb) * mb).sum() / mb.sum().clamp(min=1)
        prob = torch.softmax(logits, 1)
        for k in (1, 2):
            p = prob[:, k] * mb
            t = (yb == k).float() * mb
            tp = (p * t).sum()
            fp = (p * (1 - t)).sum()
            fn = ((1 - p) * t).sum()
            loss = loss + (1 - (tp + 1) / (tp + ALPHA * fp + BETA * fn + 1))
        loss.backward()
        opt.step()
        total += float(loss) * len(i)
    if epoch % 10 == 9:
        print("      epoch %2d  loss %.4f" % (epoch + 1, total / n))
 
torch.save({"state": net.state_dict(), "mu": MU, "sd": SD}, "outputs/unet.pt")

      class weights [ 1.   18.58 10.16]
      epoch 10  loss 2.6411
      epoch 20  loss 2.4274
      epoch 30  loss 2.2203
      epoch 40  loss 2.0181


# 5. predicting and scoring

In [75]:
net.eval()
PROB, metrics = {}, []
 
for city in CITIES:
    dd = DATA[city]
    image, label = dd["image"], dd["label"]
    height, width = label.shape
    keep = (dd["inside"] & (dd["checker"] == 1)) if city == HOME else dd["inside"]
    kind = "spatial holdout" if city == HOME else "transfer"
 
    for scheme in ("global", "per-city"):
        if scheme == "global":
            mu, sd = MU, SD
        else:
            flat = image.reshape(image.shape[0], -1)
            mu, sd = flat.mean(1), flat.std(1) + 1e-6
        norm = ((image - mu[:, None, None]) / sd[:, None, None]).astype(np.float32)
 
        acc = np.zeros((3, height, width), np.float32)
        cnt = np.zeros((height, width), np.float32)
        rr = sorted(set(list(range(0, max(height - PATCH, 0) + 1, STRIDE)) + [max(height - PATCH, 0)]))
        cc = sorted(set(list(range(0, max(width - PATCH, 0) + 1, STRIDE)) + [max(width - PATCH, 0)]))
        with torch.no_grad():
            for r in rr:
                tiles = np.stack([norm[:, r:r + PATCH, c:c + PATCH] for c in cc])
                out = torch.softmax(net(torch.from_numpy(tiles).to(device)), 1).cpu().numpy()
                for k, c in enumerate(cc):
                    acc[:, r:r + PATCH, c:c + PATCH] += out[k]
                    cnt[r:r + PATCH, c:c + PATCH] += 1
        prob = acc / np.maximum(cnt, 1)
        pred = prob.argmax(0)
        pred[~dd["inside"]] = 0
        PROB[(city, scheme)] = {"prob": prob, "pred": pred}
 
        truth, guess = label[keep], pred[keep]
        f1 = f1_score(truth, guess, labels=[0, 1, 2], average=None, zero_division=0)
        iou = jaccard_score(truth, guess, labels=[0, 1, 2], average=None, zero_division=0)
        row = {"city": city, "kind": kind, "model": "u-net", "normalization": scheme,
               "f1_newdev": f1[1], "f1_decay": f1[2],
               "iou_newdev": iou[1], "iou_decay": iou[2]}
        for k, name in ((1, "newdev"), (2, "decay")):
            binary = (truth == k).astype(int)
            score = prob[k][keep]
            row["ap_" + name] = average_precision_score(binary, score) if binary.sum() else np.nan
            row["prevalence_" + name] = binary.mean()
            row["predicted_km2_" + name] = (guess == k).sum() * 0.0009
            row["observed_km2_" + name] = binary.sum() * 0.0009
        metrics.append(row)
        print("      %-13s %-8s  AP newdev %.3f  AP decay %.3f  F1 decay %.3f"
              % (city, scheme, row["ap_newdev"], row["ap_decay"], row["f1_decay"]))
        if scheme == "per-city":
            np.savetxt("outputs/confusion_%s.csv" % city.lower(),
                       confusion_matrix(truth, guess, labels=[0, 1, 2]), fmt="%d", delimiter=",")

      Philadelphia  global    AP newdev 0.153  AP decay 0.061  F1 decay 0.107
      Philadelphia  per-city  AP newdev 0.159  AP decay 0.052  F1 decay 0.104
      Detroit       global    AP newdev 0.005  AP decay 0.030  F1 decay 0.056
      Detroit       per-city  AP newdev 0.002  AP decay 0.034  F1 decay 0.060
      Atlanta       global    AP newdev 0.179  AP decay 0.018  F1 decay 0.026
      Atlanta       per-city  AP newdev 0.107  AP decay 0.017  F1 decay 0.029


# 6. spectral null baselines

In [76]:
for city in CITIES:
    dd = DATA[city]
    label, image = dd["label"], dd["image"]
    keep = (dd["inside"] & (dd["checker"] == 1)) if city == HOME else dd["inside"]
    kind = "spatial holdout" if city == HOME else "transfer"
 
    greening = image[NDVI_T1] - image[NDVI_T0]
    building = image[NDBI_T1] - image[NDBI_T0]
    row = {"city": city, "kind": kind, "model": "spectral null", "normalization": "none"}
    for k, name, score, domain in ((1, "newdev", building, dd["convertible"]),
                                   (2, "decay", greening, dd["developed_early"])):
        mask = keep & domain
        binary = (label[mask] == k).astype(int)
        if binary.sum() == 0:
            row["ap_" + name] = np.nan
            row["f1_" + name] = np.nan
            continue
        values = score[mask]
        row["ap_" + name] = average_precision_score(binary, values)
        cut = np.partition(values, -int(binary.sum()))[-int(binary.sum())]
        row["f1_" + name] = f1_score(binary, (values >= cut).astype(int))
    metrics.append(row)
    print("      %-13s null      AP newdev %.3f  AP decay %.3f"
          % (city, row.get("ap_newdev", np.nan), row.get("ap_decay", np.nan)))
 
table = pd.DataFrame(metrics)
table.to_csv("outputs/metrics.csv", index=False)

      Philadelphia  null      AP newdev 0.407  AP decay 0.026
      Detroit       null      AP newdev 0.078  AP decay 0.022
      Atlanta       null      AP newdev 0.444  AP decay 0.019


# 7. pulling independent Philadelphia ground truth

In [104]:
POINTS = {}
 
probe = ("SELECT service_name, count(*) AS n FROM public_cases_fc "
         "WHERE requested_datetime >= '%d-01-01' AND requested_datetime < '%d-01-01' "
         "AND (service_name ILIKE '%%vacant%%' OR service_name ILIKE '%%abandon%%' "
         "OR service_name ILIKE '%%dangerous%%') GROUP BY service_name" % (T1 - 1, T1))
resp = requests.get(CARTO, params={"q": probe, "format": "csv"}, timeout=600)
print("      311 probe HTTP %d" % resp.status_code)
if resp.status_code == 200:
    names = pd.read_csv(io.StringIO(resp.text))["service_name"].dropna().tolist()
    quoted = ", ".join("'" + x.replace("'", "''") + "'" for x in names)
    frames = []
    for year in range(T0, T1 + 1):
        q = ("SELECT lat, lon FROM public_cases_fc WHERE lat IS NOT NULL "
             "AND service_name IN (%s) AND requested_datetime >= '%d-01-01' "
             "AND requested_datetime < '%d-01-01'" % (quoted, year, year + 1))
        r = requests.get(CARTO, params={"q": q, "format": "csv"}, timeout=600)
        if r.status_code == 200:
            frames.append(pd.read_csv(io.StringIO(r.text)))
    if frames:
        POINTS["311 vacancy"] = pd.concat(frames, ignore_index=True).dropna(subset=["lat", "lon"])
        print("      311 vacancy        %d records" % len(POINTS["311 vacancy"]))
else:
    print("      %s" % resp.text[:300])
 
demo_probe = ("SELECT permitdescription, count(*) AS n FROM permits "
              "WHERE permitissuedate >= '%d-01-01' "
              "AND permitdescription ILIKE '%%demolition%%' "
              "GROUP BY permitdescription" % T0)
resp = requests.get(CARTO, params={"q": demo_probe, "format": "csv"}, timeout=600)
print("      permits probe HTTP %d" % resp.status_code)
if resp.status_code == 200:
    kinds = pd.read_csv(io.StringIO(resp.text))
    for _, r in kinds.iterrows():
        print("         %-50s %d" % (r["permitdescription"], r["n"]))
    quoted = ", ".join("'" + x.replace("'", "''") + "'"
                       for x in kinds["permitdescription"].dropna())
    if quoted:
        q = ("SELECT ST_Y(the_geom) AS lat, ST_X(the_geom) AS lon FROM permits "
        "WHERE the_geom IS NOT NULL AND permitdescription IN (%s) "
        "AND permitissuedate >= '%d-01-01' AND permitissuedate < '%d-01-01'"
        % (quoted, T0, T1 + 1))
        r = requests.get(CARTO, params={"q": q, "format": "csv"}, timeout=900)
        if r.status_code == 200:
            POINTS["demolition permits"] = pd.read_csv(io.StringIO(r.text)).dropna(subset=["lat", "lon"])
            print("      demolition permits %d records" % len(POINTS["demolition permits"]))
        else:
            print("      permit pull HTTP %d  %s" % (r.status_code, r.text[:200]))
else:
    print("      %s" % resp.text[:300])
 
try:
    resp = requests.get(VPI_HUB, timeout=900)
    if resp.status_code == 200:
        vpi = gpd.read_file(io.BytesIO(resp.content))
        pt = vpi.set_crs("EPSG:4326", allow_override=True).geometry.representative_point()
        POINTS["vacant buildings"] = pd.DataFrame({"lat": pt.y.values, "lon": pt.x.values})
        print("      vacant buildings   %d records" % len(vpi))
    else:
        print("      VPI hub HTTP %d" % resp.status_code)
except Exception as err:
    print("      VPI hub failed: %s" % err)

      311 probe HTTP 200
      311 vacancy        298608 records
      permits probe HTTP 200
         Demolition Permit                                  4096
         DEMOLITION PERMIT                                  7355
      demolition permits 11209 records
      vacant buildings   8773 records


# 8. scale sensitivity and partial correlation

In [105]:
validation = []
CELL_GRIDS = {}
 
if POINTS:
    dd = DATA[HOME]
    label, pred = dd["label"], PROB[(HOME, "per-city")]["pred"]
    inside, transform = dd["inside"], dd["transform"]
    height, width = label.shape
    inverse = ~transform
 
    projected = {}
    for name, frame in POINTS.items():
        pts = gpd.GeoSeries(gpd.points_from_xy(frame["lon"], frame["lat"]),
                            crs="EPSG:4326").to_crs("EPSG:5070")
        projected[name] = np.array([inverse * (x, y)
                                    for x, y in zip(pts.x.values, pts.y.values)])
 
    for cell in CELL_SIZES:
        factor = cell // PIXEL_M
        rows_n, cols_n = height // factor, width // factor
        crop = (slice(0, rows_n * factor), slice(0, cols_n * factor))
        agg = {}
        for key, arr in [("area", inside.astype(np.float32)),
                         ("built", dd["developed_early"].astype(np.float32)),
                         ("observed", (label == 2).astype(np.float32)),
                         ("detected", (pred == 2).astype(np.float32))]:
            agg[key] = arr[crop].reshape(rows_n, factor, cols_n, factor).sum(axis=(1, 3))
        for name, xy in projected.items():
            counts = np.zeros((rows_n, cols_n), np.float32)
            r = (xy[:, 1] // factor).astype(int)
            c = (xy[:, 0] // factor).astype(int)
            ok = (r >= 0) & (r < rows_n) & (c >= 0) & (c < cols_n)
            np.add.at(counts, (r[ok], c[ok]), 1)
            agg[name] = counts
 
        keep = agg["area"] >= 0.5 * factor * factor
        control = rankdata(agg["built"][keep])
        for name in projected:
            y_rank = rankdata(agg[name][keep])
            ry = y_rank - np.polyval(np.polyfit(control, y_rank, 1), control)
            for source, arr in (("NLCD label", agg["observed"]),
                                ("U-Net detection", agg["detected"])):
                x_rank = rankdata(arr[keep])
                rho, p = spearmanr(x_rank, y_rank)
                rx = x_rank - np.polyval(np.polyfit(control, x_rank, 1), control)
                prho, pp = pearsonr(rx, ry)
                validation.append({"cell_m": cell, "n_cells": int(keep.sum()),
                                   "ground truth": name, "decay from": source,
                                   "spearman": rho, "p": p,
                                   "partial_spearman": prho, "partial_p": pp})
        if cell == PRIMARY_CELL:
            CELL_GRIDS = {"agg": agg, "keep": keep}
 
    vtable = pd.DataFrame(validation)
    vtable.to_csv("outputs/validation.csv", index=False)
    for _, r in vtable[vtable.cell_m == PRIMARY_CELL].iterrows():
        print("      %-19s vs %-16s rho %+.3f  partial %+.3f"
              % (r["ground truth"], r["decay from"], r["spearman"], r["partial_spearman"]))
else:
    vtable = pd.DataFrame()
    print("      no ground truth retrieved")

      311 vacancy         vs NLCD label       rho -0.233  partial -0.208
      311 vacancy         vs U-Net detection  rho -0.269  partial -0.220
      demolition permits  vs NLCD label       rho -0.065  partial -0.018
      demolition permits  vs U-Net detection  rho -0.151  partial -0.092
      vacant buildings    vs NLCD label       rho -0.141  partial -0.102
      vacant buildings    vs U-Net detection  rho -0.185  partial -0.130


# 9. exporting figure data

In [106]:
bundle = {}
for city in CITIES:
    dd = DATA[city]
    key = city.lower()
    bundle[key + "__label"] = dd["label"].astype(np.int8)
    bundle[key + "__pred"] = PROB[(city, "per-city")]["pred"].astype(np.int8)
    bundle[key + "__pred_global"] = PROB[(city, "global")]["pred"].astype(np.int8)
    bundle[key + "__prob_decay"] = PROB[(city, "per-city")]["prob"][2].astype(np.float16)
    bundle[key + "__prob_newdev"] = PROB[(city, "per-city")]["prob"][1].astype(np.float16)
    bundle[key + "__inside"] = dd["inside"]
    bundle[key + "__holdout"] = (dd["checker"] == 1)
    bundle[key + "__extent"] = np.array(dd["extent"])
 
if CELL_GRIDS:
    for name, arr in CELL_GRIDS["agg"].items():
        bundle["cell__" + name.replace(" ", "_")] = arr.astype(np.float32)
    bundle["cell__keep"] = CELL_GRIDS["keep"]
    bundle["cell__size_m"] = np.array([PRIMARY_CELL])
 
bundle["cities"] = np.array(list(CITIES))
np.savez_compressed("outputs/figure_data.npz", **bundle)
 
pd.concat([GRID[c]["shape"].assign(city=c) for c in CITIES]) \
    .to_file("outputs/boundaries.geojson", driver="GeoJSON")
 
regime = table[(table.model == "u-net") & (table.normalization == "per-city")].copy()
regime["regime_index"] = regime.predicted_km2_newdev / regime.predicted_km2_decay.clip(lower=1e-6)
regime[["city", "predicted_km2_newdev", "predicted_km2_decay", "regime_index"]] \
    .to_csv("outputs/development_regime.csv", index=False)
 
print(table.to_string(index=False, float_format="%.3f"))
print("      outputs/figure_data.npz, outputs/boundaries.geojson")

        city            kind         model normalization  f1_newdev  f1_decay  iou_newdev  iou_decay  ap_newdev  prevalence_newdev  predicted_km2_newdev  observed_km2_newdev  ap_decay  prevalence_decay  predicted_km2_decay  observed_km2_decay
Philadelphia spatial holdout         u-net        global      0.237     0.107       0.134      0.056      0.153              0.003                 0.598                0.566     0.061             0.009                2.420               1.579
Philadelphia spatial holdout         u-net      per-city      0.246     0.104       0.140      0.055      0.159              0.003                 0.341                0.566     0.052             0.009                2.172               1.579
     Detroit        transfer         u-net        global      0.021     0.056       0.010      0.029      0.005              0.000                 2.174                0.093     0.030             0.027              134.707              10.090
     Detroit        transfer

# 10. visualizations and d3.js web

In [107]:
bundle = np.load("outputs/figure_data.npz", allow_pickle=True)
CITIES = [str(c) for c in bundle["cities"]]
HOME = CITIES[0]
boundaries = gpd.read_file("outputs/boundaries.geojson")
metrics = pd.read_csv("outputs/metrics.csv")
validation = pd.read_csv("outputs/validation.csv") if os.path.exists("outputs/validation.csv") else pd.DataFrame()
regime = pd.read_csv("outputs/development_regime.csv")
 
SOURCE = ("Landsat Collection 2 surface reflectance via Microsoft Planetary Computer. "
          "Land cover from USGS Annual NLCD Collection 1.2. Municipal and census tract "
          "boundaries from US Census TIGER/Line 2024. Vacancy, demolition permits and 311 "
          "records from the City of Philadelphia. 30 m resolution, 2015 to 2025.")
 
 
def transform_of(city):
    minx, maxx, miny, maxy = bundle[city.lower() + "__extent"]
    return rasterio.transform.from_origin(minx, maxy, PIXEL_M, PIXEL_M)
 
 
def frame(fig, title, filename, top=0.82, bottom=0.115, left=0.035, right=0.965):
    fig.text(0.035, 0.968, title, ha="left", va="top", fontsize=25,
             fontweight="bold", color=INK)
    fig.text(0.035, 0.026, SOURCE, ha="left", va="bottom", fontsize=7.4,
             color=MUTE, wrap=True)
    fig.subplots_adjust(top=top, bottom=bottom, left=left, right=right, wspace=0.06)
    fig.patch.set_facecolor(PAPER)
    fig.patch.set_alpha(1.0)
    for axis in fig.axes:
        axis.patch.set_alpha(1.0)
        if axis.axison:
            axis.set_facecolor(PAPER)
    path = os.path.join("figures", filename)
    fig.savefig(path, facecolor=PAPER, edgecolor="none", transparent=False)
    plt.close(fig)
    print("      %s" % path)
 
 
print("[1/8] loading census tracts")
 
TRACTS = {}
for city in CITIES:
    tracts = gpd.read_file(TIGER_TRACT % STATE_OF[city]).to_crs("EPSG:5070")
    city_geom = boundaries[boundaries.city == city].geometry.union_all().buffer(0)
    hit = tracts[tracts.intersects(city_geom)].copy()
    hit["geometry"] = hit.geometry.buffer(0).intersection(city_geom)
    hit = hit[~hit.geometry.is_empty]
    hit = hit.explode(index_parts=False)
    hit = hit[hit.geom_type == "Polygon"]
    before = len(hit)
    hit = hit[hit.geometry.area >= SLIVER_M2]
    hit = hit.dissolve(by="GEOID", as_index=False)
    hit = hit[~hit.geometry.is_empty]
    TRACTS[city] = hit.reset_index(drop=True)
    print("      %-13s %d tracts, %d clip slivers dropped"
          % (city, len(hit), before - len(hit)))
 
 
print("[2/8] loading vacancy records")
 
POINTS = {}
if os.path.exists("outputs/points_cache.npz"):
    cached = np.load("outputs/points_cache.npz")
    for key in cached.files:
        POINTS[key.replace("_", " ")] = cached[key]
    print("      cache hit, %s" % ", ".join(POINTS))
else:
    collected = {}
    probe = ("SELECT service_name FROM public_cases_fc WHERE requested_datetime >= "
             "'%d-01-01' AND requested_datetime < '%d-01-01' AND (service_name ILIKE "
             "'%%vacant%%' OR service_name ILIKE '%%abandon%%' OR service_name ILIKE "
             "'%%dangerous%%') GROUP BY service_name" % (T1 - 1, T1))
    r = requests.get(CARTO, params={"q": probe, "format": "csv"}, timeout=600)
    if r.status_code == 200:
        names = pd.read_csv(io.StringIO(r.text))["service_name"].dropna().tolist()
        quoted = ", ".join("'" + x.replace("'", "''") + "'" for x in names)
        parts = []
        for year in range(T0, T1 + 1):
            q = ("SELECT lat, lon FROM public_cases_fc WHERE lat IS NOT NULL AND "
                 "service_name IN (%s) AND requested_datetime >= '%d-01-01' AND "
                 "requested_datetime < '%d-01-01'" % (quoted, year, year + 1))
            rr = requests.get(CARTO, params={"q": q, "format": "csv"}, timeout=600)
            if rr.status_code == 200:
                parts.append(pd.read_csv(io.StringIO(rr.text)))
            print("      311 %d" % year)
        if parts:
            collected["311 vacancy"] = pd.concat(parts, ignore_index=True).dropna()
 
    dp = ("SELECT permitdescription FROM permits WHERE permitissuedate >= '%d-01-01' "
          "AND permitdescription ILIKE '%%demolition%%' GROUP BY permitdescription" % T0)
    r = requests.get(CARTO, params={"q": dp, "format": "csv"}, timeout=600)
    if r.status_code == 200:
        kinds = pd.read_csv(io.StringIO(r.text))["permitdescription"].dropna()
        quoted = ", ".join("'" + x.replace("'", "''") + "'" for x in kinds)
        if quoted:
            q = ("SELECT ST_Y(the_geom) AS lat, ST_X(the_geom) AS lon FROM permits "
            "WHERE the_geom IS NOT NULL AND permitdescription IN (%s) "
             "AND permitissuedate >= '%d-01-01' AND permitissuedate < '%d-01-01'"
     % (quoted, T0, T1 + 1))
            rr = requests.get(CARTO, params={"q": q, "format": "csv"}, timeout=900)
            if rr.status_code == 200:
                collected["demolition permits"] = pd.read_csv(io.StringIO(rr.text)).dropna()
 
    try:
        r = requests.get(VPI_HUB, timeout=900)
        if r.status_code == 200:
            vpi = gpd.read_file(io.BytesIO(r.content))
            pt = vpi.set_crs("EPSG:4326", allow_override=True).geometry.representative_point()
            collected["vacant buildings"] = pd.DataFrame({"lat": pt.y.values, "lon": pt.x.values})
    except Exception as err:
        print("      VPI failed: %s" % err)
 
    for name, frame_df in collected.items():
        POINTS[name] = frame_df[["lat", "lon"]].to_numpy(np.float64)
        print("      %-20s %d records" % (name, len(frame_df)))
    if POINTS:
        np.savez_compressed("outputs/points_cache.npz",
                            **{k.replace(" ", "_"): v for k, v in POINTS.items()})
 
 
print("[3/8] aggregating to census tracts")
 
tract_stats = None
if POINTS:
    tracts = TRACTS[HOME].copy()
    transform = transform_of(HOME)
    inside = bundle[HOME.lower() + "__inside"]
    label = bundle[HOME.lower() + "__label"]
    pred = bundle[HOME.lower() + "__pred"]
 
    index = rasterize([(g, i + 1) for i, g in enumerate(tracts.geometry)],
                      out_shape=inside.shape, transform=transform, fill=0,
                      dtype="int32")
    flat = index.ravel()
    n = len(tracts) + 1
    land = np.bincount(flat, weights=inside.ravel().astype(float), minlength=n)[1:]
    tracts["land_km2"] = land * 0.0009
    tracts["observed"] = np.bincount(flat, weights=(label == 2).ravel().astype(float), minlength=n)[1:] * 0.0009
    tracts["detected"] = np.bincount(flat, weights=(pred == 2).ravel().astype(float), minlength=n)[1:] * 0.0009
 
    for name, xy in POINTS.items():
        pts = gpd.GeoSeries(gpd.points_from_xy(xy[:, 1], xy[:, 0]),
                            crs="EPSG:4326").to_crs("EPSG:5070")
        cols, rows = ~transform * (pts.x.values, pts.y.values)
        rows = rows.astype(int)
        cols = cols.astype(int)
        ok = (rows >= 0) & (rows < inside.shape[0]) & (cols >= 0) & (cols < inside.shape[1])
        counts = np.bincount(index[rows[ok], cols[ok]], minlength=n)[1:]
        tracts[name] = counts.astype(float)
 
    keepers = tracts["land_km2"] > 0.02
    tracts = tracts[keepers].copy()
    for col in ["observed", "detected"] + list(POINTS):
        tracts[col + "_density"] = tracts[col] / tracts["land_km2"]
    tract_stats = tracts
    tracts.drop(columns="geometry").to_csv("outputs/tract_summary.csv", index=False)
    print("      %d tracts with land area" % len(tracts))
 
 
def popart_map(ax, city, layers, tint, show_tracts=True):
    key = city.lower()
    inside = bundle[key + "__inside"]
    extent = bundle[key + "__extent"]
    ax.set_facecolor(tint)
    ax.imshow(np.where(inside, 0.0, np.nan), extent=extent,
              cmap=ListedColormap([tint]), interpolation="nearest", zorder=1)
 
    minx, maxx, miny, maxy = extent
    xs = np.arange(minx + DOT_SPACING_M / 2, maxx, DOT_SPACING_M)
    ys = np.arange(miny + DOT_SPACING_M / 2, maxy, DOT_SPACING_M)
    gx, gy = np.meshgrid(xs, ys)
    height, width = inside.shape
    col = ((gx - minx) / (maxx - minx) * width).astype(int).clip(0, width - 1)
    row = ((maxy - gy) / (maxy - miny) * height).astype(int).clip(0, height - 1)
    hit = inside[row, col]
    ax.scatter(gx[hit], gy[hit], s=DOT_SIZE, c=DOT, marker="o", linewidths=0, zorder=2)
 
    if show_tracts:
        TRACTS[city].boundary.plot(ax=ax, color=TRACT_LINE, linewidth=0.45, zorder=3)
 
    for arr, colour in layers:
        ax.imshow(np.where(arr, 1.0, np.nan), extent=extent,
                  cmap=ListedColormap([colour]), interpolation="nearest", zorder=4)
 
    boundaries[boundaries.city == city].boundary.plot(
        ax=ax, color=INK, linewidth=STROKE, zorder=5)
    ax.set_axis_off()
 
 
print("[4/8] detected land change, three cities")
 
fig, axes = plt.subplots(1, 3, figsize=(17, 8.6))
for ax, city, tint in zip(axes, CITIES, PANEL_TINTS):
    pred = bundle[city.lower() + "__pred"]
    popart_map(ax, city, [(pred == 1, NEWDEV_C), (pred == 2, DECAY_C)], tint)
    ax.set_title(city.upper(), loc="left", fontsize=14, fontweight="bold", pad=8)
fig.legend(handles=[Patch(facecolor=NEWDEV_C, edgecolor=INK, linewidth=1.6,
                          label="NEW DEVELOPMENT"),
                    Patch(facecolor=DECAY_C, edgecolor=INK, linewidth=1.6, label="DECAY"),
                    plt.Line2D([], [], color=TRACT_LINE, linewidth=1.2, label="CENSUS TRACT")],
           loc="lower left", bbox_to_anchor=(0.035, 0.058), ncol=3,
           prop={"weight": "bold", "size": 10}, handlelength=1.6, handleheight=1.2)
frame(fig, "DETECTED NEW DEVELOPMENT AND DECAY, 2015 TO 2025",
      "fig1_detected_change.png", top=0.80, bottom=0.125)
 
 
print("[5/8] observed against detected")
 
key = HOME.lower()
label = bundle[key + "__label"]
pred = bundle[key + "__pred"]
holdout = bundle[key + "__holdout"]
 
fig, axes = plt.subplots(1, 2, figsize=(13, 8.6))
for ax, arr, name in zip(axes, [label, pred], ["OBSERVED", "DETECTED"]):
    popart_map(ax, HOME, [((arr == 1) & holdout, NEWDEV_C),
                          ((arr == 2) & holdout, DECAY_C)], PANEL_TINTS[0])
    ax.set_title(name, loc="left", fontsize=14, fontweight="bold", pad=8)
frame(fig, "OBSERVED VERSUS DETECTED CHANGE ON HELD-OUT BLOCKS",
      "fig2_observed_vs_detected.png", top=0.80, bottom=0.09)
 
 
print("[6/8] decay detection performance")
 
pivot = metrics.pivot_table(index="city", columns=["model", "normalization"],
                            values="ap_decay").reindex(CITIES)
series = [(" ".join(c).replace("none", "").strip().upper(), pivot[c].values)
          for c in pivot.columns]
colours = SERIES[:len(series)]
 
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(len(pivot))[::-1]
h = 0.8 / len(series)
for i, ((name, values), colour) in enumerate(zip(series, colours)):
    pos = y + (len(series) - 1 - 2 * i) * h / 2
    ax.barh(pos, values, h * 0.9, label=name, color=colour,
            edgecolor=INK, linewidth=STROKE)
    for yy, v in zip(pos, values):
        ax.text(v + pivot.values.max() * 0.012, yy, "%.3f" % v, va="center",
                ha="left", fontsize=10, fontweight="bold")
ax.set_yticks(y)
ax.set_yticklabels([c.upper() for c in pivot.index], fontweight="bold")
ax.set_xlabel("AVERAGE PRECISION, DECAY CLASS")
ax.set_xlim(0, pivot.values.max() * 1.16)
ax.spines[["top", "right"]].set_visible(False)
fig.legend(*ax.get_legend_handles_labels(), loc="upper left",
           bbox_to_anchor=(0.035, 0.885), ncol=3,
           prop={"weight": "bold", "size": 9.5}, handlelength=1.4, handleheight=1.2)
frame(fig, "DECAY DETECTION PERFORMANCE BY MODEL AND CITY",
      "fig3_detection_performance.png", top=0.79, bottom=0.16, left=0.135)
 
 
print("[7/8] correlation against aggregation scale")
 
if len(validation):
    fig, ax = plt.subplots(figsize=(12, 7))
    subset = validation[validation["decay from"] == "U-Net detection"]
    palette = SERIES
    for (name, group), colour in zip(subset.groupby("ground truth"), palette):
        group = group.sort_values("cell_m")
        ax.plot(group.cell_m, group.spearman, marker="o", markersize=9, linewidth=3.2,
                color=colour, markeredgecolor=INK, markeredgewidth=1.6,
                label=name.upper() + ", BIVARIATE")
        ax.plot(group.cell_m, group.partial_spearman, marker="s", markersize=8,
                linewidth=3.2, linestyle=(0, (3, 2)), color=colour,
                markeredgecolor=INK, markeredgewidth=1.6,
                label=name.upper() + ", CONTROLLING FOR BUILT AREA")
    ax.axhline(0, color=INK, linewidth=STROKE)
    sizes = sorted(subset.cell_m.unique())
    ax.set_xscale("log")
    ax.set_xticks(sizes)
    ax.set_xticklabels([str(s) for s in sizes], fontweight="bold")
    ax.set_xlabel("AGGREGATION CELL SIZE, METRES")
    ax.set_ylabel("SPEARMAN CORRELATION, DETECTED DECAY\nAGAINST RECORDED VACANCY")
    ax.spines[["top", "right"]].set_visible(False)
    fig.legend(*ax.get_legend_handles_labels(), loc="upper left",
               bbox_to_anchor=(0.035, 0.895), ncol=2,
               prop={"weight": "bold", "size": 8.5}, handlelength=1.6, handleheight=1.1)
    frame(fig, "MODIFIABLE AREAL UNIT EFFECT ON THE DECAY CORRELATION",
          "fig4_scale_sensitivity.png", top=0.755, bottom=0.16, left=0.135)
 
 
print("[8/8] tract choropleths and change area")
 
if tract_stats is not None:
    columns = [("detected_density", "DETECTED DECAY"),
               ("observed_density", "NLCD DECAY LABEL")]
    columns += [(nm + "_density", nm.upper()) for nm in POINTS]
    cmap = ListedColormap(SEQUENTIAL)
 
    fig, axes = plt.subplots(1, len(columns), figsize=(4.9 * len(columns), 8.4))
    axes = np.atleast_1d(axes)
    for ax, (col, name) in zip(axes, columns):
        values = tract_stats[col].to_numpy()
        positive = values[values > 0]
        cuts = np.quantile(positive, [0.2, 0.4, 0.6, 0.8]) if len(positive) > 4 else [1, 2, 3, 4]
        edges = np.unique(np.concatenate([[-1e-9], cuts, [values.max() + 1]]))
        while len(edges) < 6:
            edges = np.append(edges, edges[-1] + 1)
        tract_stats.plot(ax=ax, column=col, cmap=cmap,
                         norm=BoundaryNorm(edges[:6], 5),
                         edgecolor=MUTE, linewidth=0.3)
        boundaries[boundaries.city == HOME].boundary.plot(
            ax=ax, color=INK, linewidth=STROKE)
        ax.set_axis_off()
        ax.set_title(name, loc="left", fontsize=13, fontweight="bold", pad=8)
    fig.legend(handles=[Patch(facecolor=SEQUENTIAL[i], edgecolor=INK, linewidth=1.3,
                              label=["LOWEST FIFTH", "SECOND", "THIRD", "FOURTH",
                                     "HIGHEST FIFTH"][i]) for i in range(5)],
               loc="lower left", bbox_to_anchor=(0.035, 0.055), ncol=5,
               prop={"weight": "bold", "size": 9.5}, handlelength=1.5, handleheight=1.2)
    frame(fig, "DENSITY PER SQUARE KILOMETRE BY CENSUS TRACT, PHILADELPHIA",
          "fig5_tract_comparison.png", top=0.80, bottom=0.12)
 
order = regime.set_index("city").reindex(CITIES)
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(len(order))[::-1]
top_value = max(order.predicted_km2_newdev.max(), order.predicted_km2_decay.max())
for offset, col, colour, name in ((0.19, "predicted_km2_newdev", NEWDEV_C, "NEW DEVELOPMENT"),
                                  (-0.19, "predicted_km2_decay", DECAY_C, "DECAY")):
    ax.barh(y + offset, order[col], 0.34, label=name, color=colour,
            edgecolor=INK, linewidth=STROKE)
    for yy, v in zip(y + offset, order[col]):
        ax.text(v + top_value * 0.012, yy, "%.2f km\u00b2" % v, va="center",
                ha="left", fontsize=10, fontweight="bold")
ax.set_yticks(y)
ax.set_yticklabels([c.upper() for c in order.index], fontweight="bold")
ax.set_xlabel("AREA DETECTED, SQUARE KILOMETRES")
ax.set_xlim(0, top_value * 1.2)
ax.spines[["top", "right"]].set_visible(False)
fig.legend(*ax.get_legend_handles_labels(), loc="upper left",
           bbox_to_anchor=(0.035, 0.885), ncol=2,
           prop={"weight": "bold", "size": 10}, handlelength=1.4, handleheight=1.2)
frame(fig, "DETECTED CHANGE AREA BY CITY AND CLASS",
      "fig6_change_area.png", top=0.79, bottom=0.16, left=0.135)

[1/8] loading census tracts
      Philadelphia  408 tracts, 0 clip slivers dropped
      Detroit       276 tracts, 0 clip slivers dropped
      Atlanta       192 tracts, 29 clip slivers dropped
[2/8] loading vacancy records
      311 2015
      311 2016
      311 2017
      311 2018
      311 2019
      311 2020
      311 2021
      311 2022
      311 2023
      311 2024
      311 2025
      311 vacancy          298608 records
      demolition permits   11209 records
      vacant buildings     8773 records
[3/8] aggregating to census tracts
      408 tracts with land area
[4/8] detected land change, three cities
      figures/fig1_detected_change.png
[5/8] observed against detected
      figures/fig2_observed_vs_detected.png
[6/8] decay detection performance
      figures/fig3_detection_performance.png
[7/8] correlation against aggregation scale
      figures/fig4_scale_sensitivity.png
[8/8] tract choropleths and change area
      figures/fig5_tract_comparison.png
      figures/fig6_ch

# 11. annual land cover trajectory

In [100]:
LNDCOV_WMS = "https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Native_conus_year_data/wms"
LNDCOV_LAYER = "Land-Cover-Native_conus_year_data"
NLCD_RGB = {(70, 107, 159): 11, (209, 222, 248): 12, (222, 197, 197): 21,
            (217, 146, 130): 22, (235, 0, 0): 23, (171, 0, 0): 24,
            (179, 172, 159): 31, (104, 171, 95): 41, (28, 95, 44): 42,
            (181, 197, 143): 43, (204, 184, 121): 52, (223, 223, 194): 71,
            (220, 217, 57): 81, (171, 108, 40): 82, (184, 217, 235): 90,
            (108, 159, 184): 95}
INTENSITY = {21: 1, 22: 2, 23: 3, 24: 4}
TILE = 1800
SERIES_YEARS = list(range(T0, T1 + 1))

if os.path.exists("outputs/annual_cover.npz"):
    annual = dict(np.load("outputs/annual_cover.npz"))
    print("      cache hit, %d rasters" % len(annual))
else:
    annual = {}
    for city in CITIES:
        inside = bundle[city.lower() + "__inside"]
        height, width = inside.shape
        minx, maxx, miny, maxy = bundle[city.lower() + "__extent"]
        for year in SERIES_YEARS:
            canvas = None
            for r0 in range(0, height, TILE):
                for c0 in range(0, width, TILE):
                    h, w = min(TILE, height - r0), min(TILE, width - c0)
                    bx0, by1 = minx + c0 * PIXEL_M, maxy - r0 * PIXEL_M
                    resp = requests.get(LNDCOV_WMS, params={
                        "service": "WMS", "version": "1.1.1", "request": "GetMap",
                        "layers": LNDCOV_LAYER, "styles": "", "srs": "EPSG:5070",
                        "bbox": "%f,%f,%f,%f" % (bx0, by1 - h * PIXEL_M,
                                                 bx0 + w * PIXEL_M, by1),
                        "width": w, "height": h, "format": "image/geotiff",
                        "transparent": "false",
                        "time": "%d-01-01T00:00:00.000Z" % year}, timeout=300)
                    resp.raise_for_status()
                    with rasterio.MemoryFile(resp.content) as mem:
                        with mem.open() as src:
                            block = src.read()
                    if canvas is None:
                        canvas = np.zeros((block.shape[0], height, width), block.dtype)
                    canvas[:, r0:r0 + h, c0:c0 + w] = block[:, :h, :w]
            if canvas.shape[0] == 1:
                arr = canvas[0].astype(np.uint8)
            else:
                arr = np.zeros((height, width), np.uint8)
                rgb = canvas[:3].transpose(1, 2, 0)
                for colour, code in NLCD_RGB.items():
                    arr[np.all(rgb == np.array(colour, canvas.dtype), axis=-1)] = code
            annual["%s__%d" % (city.lower(), year)] = arr
            print("      %-13s %d" % (city, year))
    np.savez_compressed("outputs/annual_cover.npz", **annual)

rows = []
for city in CITIES:
    inside = bundle[city.lower() + "__inside"]
    base_mask, base_area = None, None
    for year in SERIES_YEARS:
        cover = annual["%s__%d" % (city.lower(), year)]
        rank = np.zeros(cover.shape, np.int8)
        for code, level in INTENSITY.items():
            rank[cover == code] = level
        developed = (rank >= 1) & inside
        area = developed.sum() * 0.0009
        if base_mask is None:
            base_mask, base_area = developed.copy(), max(area, 1e-9)
        rows.append({"city": city, "year": year, "developed_km2": area,
                     "developed_index": 100.0 * area / base_area,
                     "mean_intensity": float(rank[base_mask].mean())})

trajectory = pd.DataFrame(rows)
trajectory.to_csv("outputs/trajectory.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.6))
panels = [("developed_index", "DEVELOPED AREA, %d = 100" % T0),
          ("mean_intensity", "MEAN INTENSITY OF %d DEVELOPED PIXELS" % T0)]
for ax, (col, ylabel) in zip(axes, panels):
    for city, colour in zip(CITIES, SERIES):
        part = trajectory[trajectory.city == city].sort_values("year")
        ax.plot(part.year, part[col], marker="o", markersize=7, linewidth=3.0,
                color=colour, markeredgecolor=INK, markeredgewidth=1.4,
                label=city.upper())
    ax.set_xlabel("YEAR")
    ax.set_ylabel(ylabel)
    ax.set_xticks(SERIES_YEARS[::2])
    ax.spines[["top", "right"]].set_visible(False)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper left",
           bbox_to_anchor=(0.035, 0.885), ncol=3,
           prop={"weight": "bold", "size": 10}, handlelength=1.6, handleheight=1.1)
frame(fig, "ANNUAL DEVELOPED AREA AND INTENSITY, %d TO %d" % (T0, T1),
      "fig7_trajectory.png", top=0.775, bottom=0.16, left=0.085)

      Philadelphia  2015
      Philadelphia  2016
      Philadelphia  2017
      Philadelphia  2018
      Philadelphia  2019
      Philadelphia  2020
      Philadelphia  2021
      Philadelphia  2022
      Philadelphia  2023
      Philadelphia  2024
      Philadelphia  2025
      Detroit       2015
      Detroit       2016
      Detroit       2017
      Detroit       2018
      Detroit       2019
      Detroit       2020
      Detroit       2021
      Detroit       2022
      Detroit       2023
      Detroit       2024
      Detroit       2025
      Atlanta       2015
      Atlanta       2016
      Atlanta       2017
      Atlanta       2018
      Atlanta       2019
      Atlanta       2020
      Atlanta       2021
      Atlanta       2022
      Atlanta       2023
      Atlanta       2024
      Atlanta       2025
      figures/fig7_trajectory.png


In [103]:
if tract_stats is not None:
    names = list(POINTS)
    fig, axes = plt.subplots(1, len(names), figsize=(5.6 * len(names), 6.6))
    axes = np.atleast_1d(axes)
    for ax, name, colour in zip(axes, names, SERIES):
        x = tract_stats["detected_density"].to_numpy(float)
        y = tract_stats[name + "_density"].to_numpy(float)
        ok = np.isfinite(x) & np.isfinite(y)
        rho, p = spearmanr(x[ok], y[ok])
        ax.scatter(x[ok], y[ok], s=30, color=colour, edgecolor=INK,
                   linewidths=0.8, alpha=0.85, zorder=3)
        slope, intercept = np.polyfit(x[ok], y[ok], 1)
        grid = np.linspace(x[ok].min(), x[ok].max(), 50)
        ax.plot(grid, slope * grid + intercept, color=INK, linewidth=2.6, zorder=4)
        ax.text(0.97, 0.95, "SPEARMAN %+.3f\np %.1e\nn = %d" % (rho, p, ok.sum()),
                transform=ax.transAxes, ha="right", va="top",
                fontsize=10, fontweight="bold", color=INK)
        ax.set_title(name.upper(), loc="left", fontsize=13, fontweight="bold", pad=8)
        ax.set_xlabel("DETECTED DECAY, SHARE OF TRACT LAND")
        ax.set_ylabel("RECORDS PER SQUARE KILOMETRE")
        ax.spines[["top", "right"]].set_visible(False)
    frame(fig, "TRACT-LEVEL RELATIONSHIP BETWEEN DETECTED DECAY AND RECORDED VACANCY",
          "fig8_tract_scatter.png", top=0.80, bottom=0.155, left=0.055)

      figures/fig8_tract_scatter.png


In [102]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.6))
for ax, k, name, field in [(axes[0], 1, "NEW DEVELOPMENT", "__prob_newdev"),
                           (axes[1], 2, "DECAY", "__prob_decay")]:
    for city, colour in zip(CITIES, SERIES):
        key = city.lower()
        inside = bundle[key + "__inside"]
        mask = (inside & bundle[key + "__holdout"]) if city == HOME else inside
        truth = (bundle[key + "__label"][mask] == k).astype(int)
        score = bundle[key + field][mask].astype(np.float32)
        if truth.sum() == 0:
            continue
        precision, recall, _ = precision_recall_curve(truth, score)
        ap = average_precision_score(truth, score)
        ax.plot(recall, precision, linewidth=3.0, color=colour,
                label="%s, AP %.3f" % (city.upper(), ap))
        ax.axhline(truth.mean(), color=colour, linewidth=1.3, linestyle=(0, (2, 2)))
    ax.set_title(name, loc="left", fontsize=13, fontweight="bold", pad=8)
    ax.set_xlabel("RECALL")
    ax.set_ylabel("PRECISION")
    ax.set_xlim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(prop={"weight": "bold", "size": 9.5})
frame(fig, "PRECISION AND RECALL AGAINST CLASS PREVALENCE",
      "fig9_precision_recall.png", top=0.80, bottom=0.155, left=0.075)

      figures/fig9_precision_recall.png
